# Solutions · 03 · Entropy, refrigeration and heat pumps

Worked solutions to the exercises of [notebook 03](../notebooks/03_entropy_refrigeration_and_heat_pumps.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engthermo import cycles, tables
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A cold store must hold -25 degC with ammonia condensing at 35 degC. Compute the COP, the compressor discharge temperature and the pressure ratio, and compare with a two-stage arrangement (ideal intercooling at the geometric-mean pressure - treat it as two cycles in series with an intermediate temperature of 0 degC).

In [2]:
one = cycles.vapor_compression("ammonia", 248.15, 308.15)
lo = cycles.vapor_compression("ammonia", 248.15, 273.15)              # low stage: -25 -> 0 degC
hi = cycles.vapor_compression("ammonia", 273.15, 308.15)              # high stage: 0 -> 35 degC
# two cycles in series: the high stage must absorb the low stage's condenser heat
w_two = lo["w_compressor"] + hi["w_compressor"] * lo["q_condenser"] / hi["q_evaporator"]
print(f"single stage: COP {one['COP_cooling']:.2f}, discharge {one['states']['2']['T']-273.15:.0f} degC, pressure ratio {one['p_high']/one['p_low']:.1f}")
print(f"two stages:   COP {lo['q_evaporator']/w_two:.2f}, discharge temperatures {lo['states']['2']['T']-273.15:.0f} and {hi['states']['2']['T']-273.15:.0f} degC, "
      f"pressure ratios {lo['p_high']/lo['p_low']:.1f} and {hi['p_high']/hi['p_low']:.1f}")

single stage: COP 3.21, discharge 135 degC, pressure ratio 8.9
two stages:   COP 3.62, discharge temperatures 43 and 81 degC, pressure ratios 2.8 and 3.1


Splitting the compression roughly halves each stage's pressure ratio, cuts the discharge temperature from
well above 100 degC to values the compressor oil tolerates, and improves the COP. Large industrial ammonia
plants for cold stores are two-stage for exactly these reasons.

## Exercise 2

For the propane heat pump, compare the seasonal energy use of radiators (condenser 55 degC) with underfloor heating (condenser 35 degC) for a house needing 12 000 kWh of heat per winter at an average outdoor temperature of 4 degC.

In [3]:
for name, T_cond in (("radiators, 55 degC", 55), ("underfloor heating, 35 degC", 35)):
    cop = cycles.vapor_compression("propane", 4 + 273.15 - 8, T_cond + 273.15, eta_compressor=0.7)["COP_heating"]
    print(f"{name:<28}: seasonal COP {cop:.2f} -> {12000 / cop:.0f} kWh of electricity for 12 000 kWh of heat")

radiators, 55 degC          : seasonal COP 3.15 -> 3807 kWh of electricity for 12 000 kWh of heat
underfloor heating, 35 degC : seasonal COP 4.89 -> 2454 kWh of electricity for 12 000 kWh of heat


Lowering the condenser temperature by 20 K saves about a third of the electricity over the winter. The
building's heat emitters - not the heat pump - decide much of a heat-pump system's efficiency, which is why
new heat-pump installations come with large radiators or floor heating.

## Exercise 3

**Implement it yourself:** compute the entropy generated in the real compressor of the R134a cycle (80 % efficiency) from the table states, and the exergy destroyed per kilogram of refrigerant.

In [4]:
vc = cycles.vapor_compression("R134a", 253.15, 313.15, eta_compressor=0.8)
s1, s2 = vc["states"]["1"], vc["states"]["2"]
s_gen = s2["s"] - s1["s"]
print(f"real compressor (80 %): entropy generated {s_gen:.1f} J/(kg K) per kg; exergy destroyed at T0 = 298 K: {298.15 * s_gen / 1e3:.1f} kJ/kg "
      f"(of {vc['w_compressor']/1e3:.1f} kJ/kg of work)")

real compressor (80 %): entropy generated 32.5 J/(kg K) per kg; exergy destroyed at T0 = 298 K: 9.7 kJ/kg (of 53.1 kJ/kg of work)


An adiabatic compressor's entropy generation is simply the entropy rise across it; a fifth of the work input
is destroyed as exergy - the price of the 80 % isentropic efficiency, and the largest loss in the cycle.